# Complete-Profile Evaluation

Split out from `results_latex_table.ipynb` (which keeps the main per-process/per-mode process-modelling quality table). This notebook covers ONLY whole-case curve ("complete profile") evaluation: Complete-Curve Evaluation (per-case, shift-tolerant Wasserstein), the Schedule Profile Evaluation (Direct Method vs. Stochastic generators vs. the Proposed Process-Simulation Approach), and the Population-Level (unpaired) Distributional Evaluation (shape + magnitude). No process-modelling-quality metrics (Fitness/Precision/Edge F1/Duration error) or process-quality-vs-energy-quality comparisons are included here -- see `results_latex_table.ipynb` for those.

## Setup (minimal -- config, run_dir, process list; no process-quality table)

In [1]:
# ── Config ────────────────────────────────────────────────────────────────────
# Set the experiment number to load (picks the latest run automatically)
EXPERIMENT = 951
# Which evaluation split to display: 'train' or 'test'
# Both live in the same row — train_* vs test_* column prefixes.
SPLIT = 'test'

In [2]:
import pandas as pd
from pathlib import Path

results_root = Path('..') / 'results'

# Find the latest run folder for the given experiment
prefix = f'experiment_{EXPERIMENT}_'
runs = sorted([d for d in results_root.iterdir() if d.is_dir() and d.name.startswith(prefix)])
assert runs, f'No runs found for experiment {EXPERIMENT}'
run_dir = runs[-1]
print(f'Loading: {run_dir.name}')

df = pd.read_parquet(run_dir / 'process_eval_results.parquet')
# The 'split' column marks the training set used (always 'TRAIN' or 'ALL DATA').
# Train vs test evaluation results are distinguished by column prefix: train_* vs test_*.
# Check the requested prefix exists.
split_prefix = SPLIT.lower() + '_'
available_prefixes = set()
for c in df.columns:
    for p in ('train_', 'test_'):
        if c.startswith(p):
            available_prefixes.add(p.rstrip('_'))
print(f'Available column prefixes: {sorted(available_prefixes)}')
assert split_prefix.rstrip('_') in available_prefixes, \
    f'No columns with prefix "{split_prefix}" found. Available: {sorted(available_prefixes)}'

print(f'Split prefix: {split_prefix} | {len(df)} rows | processes: {sorted(df["process"].unique())}')

Loading: experiment_951_20260720_182343
Available column prefixes: ['test', 'train']
Split prefix: test_ | 60 rows | processes: ['process_1', 'process_2', 'process_3', 'process_4_1', 'process_4_2', 'process_5']


In [3]:
# ── Mode display labels (mirrors modelling.py _display_mode) ─────────────────
def display_mode(m):
    m = str(m)
    if not m.startswith('petri_net_'):
        return m
    rest = m[len('petri_net_'):]
    if rest.endswith('_ml_plus_global'):
        return rest[:-len('_ml_plus_global')] + ' / ml_global'
    if rest.endswith('_ml_plus_per_act'):
        return rest[:-len('_ml_plus_per_act')] + ' / ml_local'
    return rest + ' / baseline'

df['mode_label'] = df['mode'].map(display_mode)

In [4]:
# -- Process list (only thing needed from the main results table build) --
processes = sorted(df['process'].unique())
print(f'Processes: {processes}')

Processes: ['process_1', 'process_2', 'process_3', 'process_4_1', 'process_4_2', 'process_5']


In [5]:
import numpy as np

def shorten_sensor(name, max_len=40):
    name = str(name).replace('_energy_to_model', '').replace('_to_model', '')
    return name if len(name) <= max_len else name[:max_len - 1] + '…'

# Complete-Curve Evaluation (per-case, shift-tolerant Wasserstein)

New, additional evaluation -- does not replace anything above. For every real test case, matched to its same-`case_id` counterpart in each simulated log, this:

1. Concatenates the case's real per-activity sensor curves (in real time order) into one complete real profile.
2. Predicts each of that case's activities' curves with the trained `baseline` (median/barycenter per activity) and `exog_prev_activity` ("DTW + Ext. Factors + Prev Act", the strongest curve-fitting approach) pipelines, using the simulated log's own activities/durations, and concatenates them into one complete simulated profile for the same case.
3. Compares the two complete profiles with two Wasserstein (Earth Mover's) distances along orthogonal axes, instead of a single mean/total summary stat (which a trivial "predict the mean everywhere" model could win without getting anything useful right):
   - **W1 (time)** -- curve values as weights, case-relative time as the transport axis. Answers "is the timing/shape right." Order-sensitive; blind to overall magnitude (a right-shaped-but-30%-too-high curve still scores well here, since W1 normalises each curve to the same total mass before comparing).
   - **W1 (value)** -- raw curve values as the transport axis, unweighted. Answers "is the distribution of magnitudes right" -- catches scale *and* spread/variance errors. Order-blind (doesn't know *when* a value occurred), so it's exactly complementary to W1 (time). Works uniformly for intensive sensors (temperature, concentration) and extensive ones (power, flow), since nothing is summed or averaged before comparing.

A model can't fool both at once: winning W1 (value) by predicting a flat mean loses badly on W1 (time), since all its mass then sits in the wrong place in time.

Source: `complete_curve_eval_results/<process>/<mode>/complete_curve_summary[_exog_prev_activity].csv`, populated per (process, mode, approach) when the pipeline runs with `run_energy_modelling=True`. If it's missing, re-run the pipeline after this modelling.py update.


In [6]:
# -- Load complete-curve evaluation results (per-case, shift-tolerant Wasserstein) --
# Discovers every approach that has a complete_curve_summary*.csv in a mode folder
# (complete-curve eval runs against every trained non-seq2seq approach, not just
# baseline + exog_prev_activity -- see modelling.py's COMPLETE-CURVE EVAL block),
# rather than assuming a fixed pair.
#
# Two Wasserstein distances along two orthogonal axes, not a mean/total summary
# stat: wasserstein_time (values as weights, time as the transport axis -- is the
# timing/shape right) and wasserstein_value (raw values as the transport axis,
# unweighted -- is the distribution of magnitudes right). Neither can be gamed by
# a trivial "predict the mean everywhere" model: that scores badly on
# wasserstein_time (all its mass sits in the wrong place in time) and on
# wasserstein_value (zero variance vs. the real curve's actual spread). Works
# uniformly for intensive (temperature) and extensive (power, flow) sensors alike,
# since nothing is summed or averaged before comparing.
complete_curve_dir = run_dir / 'complete_curve_eval_results'

if not complete_curve_dir.exists():
    print(f'Not found: {complete_curve_dir}\n(run the pipeline with run_energy_modelling=True to generate it)')
    df_complete_curve = pd.DataFrame()
else:
    rows = []
    for proc_dir in sorted(complete_curve_dir.iterdir()):
        if not proc_dir.is_dir():
            continue
        for mode_dir in sorted(proc_dir.iterdir()):
            if not mode_dir.is_dir():
                continue
            for summary_path in sorted(mode_dir.glob('complete_curve_summary*.csv')):
                # 'complete_curve_summary.csv' -> baseline; '..._<approach>.csv' -> <approach>
                _stem = summary_path.stem  # 'complete_curve_summary' or 'complete_curve_summary_<approach>'
                _suffix = _stem[len('complete_curve_summary'):]
                approach = _suffix[1:] if _suffix.startswith('_') else 'baseline'

                summ = pd.read_csv(summary_path)
                if summ.empty:
                    continue
                mode_name = summ['mode'].iloc[0] if 'mode' in summ.columns else mode_dir.name
                for _, r in summ.iterrows():
                    rows.append({
                        'process':                   proc_dir.name,
                        'mode':                      mode_name,
                        'approach':                  approach,
                        'sensor':                    r['sensor'],
                        'wasserstein_time_median':   r['wasserstein_time_median'],
                        'wasserstein_value_median':  r.get('wasserstein_value_median'),
                        'n_cases':                   r['n_cases'],
                    })
    df_complete_curve = pd.DataFrame(rows)

print(f'{len(df_complete_curve)} (process, mode, approach, sensor) rows loaded')
if not df_complete_curve.empty:
    print(f'Approaches found: {sorted(df_complete_curve["approach"].unique())}')
df_complete_curve.head(20)


2700 (process, mode, approach, sensor) rows loaded
Approaches found: ['baseline', 'exog_prev_activity', 'ml_dtw_linear_decode']


,process,mode,approach,sensor,wasserstein_time_median,wasserstein_value_median,n_cases
0,process_1,petri_net_budget,baseline,autoclave_cooling_water_demand_kW_energy_to_model,13.536203,196.150275,88
1,process_1,petri_net_budget,baseline,autoclave_steam_demand_kW_energy_to_model,15.171849,364.501700,90
2,process_1,petri_net_budget,baseline,bottling_power_kW_energy_to_model,10.358559,0.191080,72
3,process_1,petri_net_budget,baseline,destillation_cooling_demand_kW_energy_to_model,6.193466,744.152886,90
4,process_1,petri_net_budget,baseline,destillation_steam_demand_kW_energy_to_model,5.716211,744.478709,90
5,process_1,petri_net_budget,baseline,individual_packaging_power_kW_energy_to_model,75.091677,1.291138,79
6,process_1,petri_net_budget,exog_prev_activity,autoclave_cooling_water_demand_kW_energy_to_model,14.289718,169.489440,90
7,process_1,petri_net_budget,exog_prev_activity,autoclave_steam_demand_kW_energy_to_model,15.060015,376.745917,90
8,process_1,petri_net_budget,exog_prev_activity,bottling_power_kW_energy_to_model,10.717442,0.207380,90
9,process_1,petri_net_budget,exog_prev_activity,destillation_cooling_demand_kW_energy_to_model,6.172123,751.850505,90


In [7]:
# -- Scale-free versions of the two complete-curve Wasserstein metrics --
# wasserstein_time_median is in minutes and wasserstein_value_median is in the
# sensor's own raw units -- neither is comparable across sensors of different
# scale (a kg/h sensor vs. a kW sensor, a short-cycle process vs. a long one),
# so aggregating them across sensors (as the combined table below does) lets
# whichever sensor has the largest units/duration dominate the median. Fix,
# mirroring the relative_wasserstein trick used for the pooled energy-
# distribution metric above: divide each by a real, model-independent scale
# reference for that (process, sensor) --
#   relative_wasserstein_time  = wasserstein_time_median  / real median case duration (minutes)
#   relative_wasserstein_value = wasserstein_value_median / real per-case-mean median (same raw units)
# Note: the pooled *instantaneous* value median (per_sensor_pooled_values.csv)
# is often exactly 0 for intermittent sensors (idle most of the time), which
# would blow up the ratio -- per_case_sensor_mean.csv's real_median (median of
# each case's own mean value) stays a well-behaved nonzero denominator.
if not df_complete_curve.empty:
    case_duration_min = {}
    for proc in df_complete_curve['process'].unique():
        log_path = run_dir / 'predicted_logs' / f'{proc}_test_set.parquet'
        if not log_path.exists():
            continue
        log_df = pd.read_parquet(log_path, columns=['case_id', 'timestamp_start', 'timestamp_end'])
        case_ends = log_df.groupby('case_id')['timestamp_end'].max()
        case_starts = log_df.groupby('case_id')['timestamp_start'].min()
        durations = (case_ends - case_starts).dt.total_seconds() / 60.0
        case_duration_min[proc] = durations.median()

    real_val_median = {}
    for proc_dir in sorted((run_dir / 'energy_distribution_results').iterdir()):
        if not proc_dir.is_dir():
            continue
        for mode_dir in sorted(proc_dir.iterdir()):
            csv_path = mode_dir / 'per_case_sensor_mean.csv'
            if not csv_path.exists():
                continue
            pooled_df = pd.read_csv(csv_path)
            for _, r in pooled_df.iterrows():
                real_val_median.setdefault((proc_dir.name, r['sensor']), r['real_median'])

    df_complete_curve['case_duration_median_min'] = df_complete_curve['process'].map(case_duration_min)
    df_complete_curve['real_value_median'] = df_complete_curve.apply(
        lambda r: real_val_median.get((r['process'], r['sensor'])), axis=1
    )
    df_complete_curve['relative_wasserstein_time'] = (
        df_complete_curve['wasserstein_time_median']
        / df_complete_curve['case_duration_median_min'].replace(0, float('nan'))
    )
    df_complete_curve['relative_wasserstein_value'] = (
        df_complete_curve['wasserstein_value_median']
        / df_complete_curve['real_value_median'].abs().replace(0, float('nan'))
    )

    print('Relative (scale-free) complete-curve metrics added.')
    display(df_complete_curve[['process', 'mode', 'approach', 'sensor',
                               'wasserstein_time_median', 'relative_wasserstein_time',
                               'wasserstein_value_median', 'relative_wasserstein_value']].head(20))
else:
    print('Nothing to show -- df_complete_curve is empty.')

Relative (scale-free) complete-curve metrics added.


,process,mode,approach,sensor,wasserstein_time_median,relative_wasserstein_time,wasserstein_value_median,relative_wasserstein_value
0,process_1,petri_net_budget,baseline,autoclave_cooling_water_demand_kW_energy_to_model,13.536203,0.103544,196.150275,1.046627
1,process_1,petri_net_budget,baseline,autoclave_steam_demand_kW_energy_to_model,15.171849,0.116056,364.501700,1.369153
2,process_1,petri_net_budget,baseline,bottling_power_kW_energy_to_model,10.358559,0.079237,0.191080,0.110061
3,process_1,petri_net_budget,baseline,destillation_cooling_demand_kW_energy_to_model,6.193466,0.047376,744.152886,4.257275
4,process_1,petri_net_budget,baseline,destillation_steam_demand_kW_energy_to_model,5.716211,0.043726,744.478709,6.016967
5,process_1,petri_net_budget,baseline,individual_packaging_power_kW_energy_to_model,75.091677,0.574407,1.291138,0.560248
6,process_1,petri_net_budget,exog_prev_activity,autoclave_cooling_water_demand_kW_energy_to_model,14.289718,0.109308,169.489440,0.904369
7,process_1,petri_net_budget,exog_prev_activity,autoclave_steam_demand_kW_energy_to_model,15.060015,0.115200,376.745917,1.415145
8,process_1,petri_net_budget,exog_prev_activity,bottling_power_kW_energy_to_model,10.717442,0.081982,0.207380,0.119450
9,process_1,petri_net_budget,exog_prev_activity,destillation_cooling_demand_kW_energy_to_model,6.172123,0.047213,751.850505,4.301313


## Per-sensor breakdown (complete-profile W1 time / W1 shape)

The combined table above already medians `relative_wasserstein_time`/`relative_wasserstein_value`
across sensors before combining with the process-quality columns. This section shows the
same two scale-free metrics **per sensor** instead, mirroring the Energy-Distribution
section's per-sensor tables above -- rows = sensor, columns = mode x curve approach
(Baseline / Best = DTW + Ext. Factors + Prev. Activity). `relative_wasserstein_time`
("W1 timing") is `wasserstein_time_median` divided by the real median case duration;
`relative_wasserstein_value` ("W1 shape") is `wasserstein_value_median` divided by the
real per-case-mean median for that sensor -- both scale-free, so a kg/h sensor and a °C
sensor can be compared fairly. Lower is better for both.


In [8]:
# -- Per-sensor breakdown: complete-profile W1 (time) / W1 (value), relative --
# rows = sensor, columns = mode x approach (Baseline / Best)
_cc_approach_labels = {'baseline': 'Baseline', 'exog_prev_activity': 'Best'}

def _cc_per_sensor_table(metric_col, metric_label, metric_desc):
    if df_complete_curve.empty:
        print('Nothing to show -- df_complete_curve is empty.')
        return
    dfc = df_complete_curve[df_complete_curve['approach'].isin(_cc_approach_labels)].copy()
    dfc['mode_label'] = dfc['mode'].map(display_mode)
    dfc['sensor_short'] = dfc['sensor'].map(shorten_sensor)
    dfc['approach_label'] = dfc['approach'].map(_cc_approach_labels)

    for proc in sorted(dfc['process'].unique()):
        sub = dfc[dfc['process'] == proc]
        pivot = sub.pivot_table(index='sensor_short', columns=['mode_label', 'approach_label'],
                                values=metric_col, aggfunc='first')
        pivot = pivot.reindex(columns=sorted(pivot.columns, key=lambda c: (c[0], c[1] != 'Best')))

        str_table = pd.DataFrame(index=pivot.index, columns=pivot.columns, dtype=object)
        for idx in pivot.index:
            vals = pivot.loc[idx].dropna()
            if vals.empty:
                str_table.loc[idx] = ''
                continue
            best_val = vals.min()
            for col in pivot.columns:
                val = pivot.loc[idx, col]
                if pd.isna(val):
                    str_table.loc[idx, col] = ''
                    continue
                fmt = f'{val:.3f}'
                str_table.loc[idx, col] = (r'\textbf{' + fmt + r'}') if abs(val - best_val) < 1e-6 else fmt

        str_table.columns = [f'{m} ({a})' for m, a in str_table.columns]
        n_cols = len(str_table.columns)
        col_format = 'l|' + '|'.join(['c'] * n_cols)
        latex = str_table.to_latex(
            escape=False, column_format=col_format,
            caption=(
                f'Complete-profile evaluation for {proc}, experiment {EXPERIMENT} ({SPLIT} evaluation). '
                f'{metric_desc} '
                r'Lower is better. \textbf{Bold} = best mode/approach per sensor.'
            ),
            label=f'tab:complete_profile_{metric_col}_{EXPERIMENT}_{SPLIT}_{proc}',
            position='H',
        )
        print(f'% === {proc}: {pivot.shape[0]} sensors x {pivot.shape[1]} mode/approach cols -- {metric_label} ===')
        print(latex.replace('_', r'\_'))
        print()

    display(
        dfc.pivot_table(index=['process', 'sensor_short'], columns=['mode_label', 'approach_label'],
                        values=metric_col, aggfunc='first').round(3)
    )

_cc_per_sensor_table(
    'relative_wasserstein_time', 'W1 (time)',
    r"W1 (time), scale-free (relative to real median case duration) --- is the timing/shape right, "
    r"as a fraction of a typical case."
)


% === process_1: 6 sensors x 20 mode/approach cols -- W1 (time) ===
\begin{table}[H]
\caption{Complete-profile evaluation for process\_1, experiment 951 (test evaluation). W1 (time), scale-free (relative to real median case duration) --- is the timing/shape right, as a fraction of a typical case. Lower is better. \textbf{Bold} = best mode/approach per sensor.}
\label{tab:complete\_profile\_relative\_wasserstein\_time\_951\_test\_process\_1}
\begin{tabular}{l|c|c|c|c|c|c|c|c|c|c|c|c|c|c|c|c|c|c|c|c}
\toprule
 & budget / baseline (Best) & budget / baseline (Baseline) & budget / ml\_global (Best) & budget / ml\_global (Baseline) & budget / ml\_local (Best) & budget / ml\_local (Baseline) & heuristic / baseline (Best) & heuristic / baseline (Baseline) & heuristic / ml\_global (Best) & heuristic / ml\_global (Baseline) & heuristic / ml\_local (Best) & heuristic / ml\_local (Baseline) & wip\_aware / baseline (Best) & wip\_aware / baseline (Baseline) & wip\_aware / ml\_global (Best) & wip\_aw

mode_label                                         budget / baseline         \
approach_label                                              Baseline   Best   
process   sensor_short                                                        
process_1 autoclave_cooling_water_demand_kW                    0.104  0.109   
          autoclave_steam_demand_kW                            0.116  0.115   
          bottling_power_kW                                    0.079  0.082   
          destillation_cooling_demand_kW                       0.047  0.047   
          destillation_steam_demand_kW                         0.044  0.043   
...                                                              ...    ...   
process_5 temp_nach_Kuehlturmkuehler_(WT6)_5s                  0.211  0.209   
          temp_nach_WR2,_vor_Druckerhoehungspumpe…             0.205  0.205   
          temp_nach_WR2_(WT2)_5s                               0.204  0.199   
          temp_vor_Vorwärmer_(WT_2)_1h                         0.204  0.202   
          vor_Vorwärmer_(WT_2)_5s                              0.205  0.202   

mode_label                                         budget / ml_global         \
approach_label                                               Baseline   Best   
process   sensor_short                                                         
process_1 autoclave_cooling_water_demand_kW                     0.095  0.097   
          autoclave_steam_demand_kW                             0.080  0.080   
          bottling_power_kW                                     0.063  0.063   
          destillation_cooling_demand_kW                        0.035  0.035   
          destillation_steam_demand_kW                          0.032  0.032   
...                                                               ...    ...   
process_5 temp_nach_Kuehlturmkuehler_(WT6)_5s                   0.360  0.359   
          temp_nach_WR2,_vor_Druckerhoehungspumpe…              0.361  0.361   
          temp_nach_WR2_(WT2)_5s                                0.354  0.354   
          temp_vor_Vorwärmer_(WT_2)_1h                          0.359  0.359   
          vor_Vorwärmer_(WT_2)_5s                               0.359  0.359   

mode_label                                         budget / ml_local         \
approach_label                                              Baseline   Best   
process   sensor_short                                                        
process_1 autoclave_cooling_water_demand_kW                    0.065  0.084   
          autoclave_steam_demand_kW                            0.025  0.025   
          bottling_power_kW                                    0.015  0.015   
          destillation_cooling_demand_kW                       0.013  0.012   
          destillation_steam_demand_kW                         0.013  0.012   
...                                                              ...    ...   
process_5 temp_nach_Kuehlturmkuehler_(WT6)_5s                  0.268  0.269   
          temp_nach_WR2,_vor_Druckerhoehungspumpe…             0.272  0.272   
          temp_nach_WR2_(WT2)_5s                               0.266  0.266   
          temp_vor_Vorwärmer_(WT_2)_1h                         0.270  0.269   
          vor_Vorwärmer_(WT_2)_5s                              0.270  0.270   

mode_label                                         heuristic / baseline  \
approach_label                                                 Baseline   
process   sensor_short                                                    
process_1 autoclave_cooling_water_demand_kW                       0.137   
          autoclave_steam_demand_kW                               0.128   
          bottling_power_kW                                       0.093   
          destillation_cooling_demand_kW                          0.048   
          destillation_steam_demand_kW                            0.044   
...                                                              

In [9]:
_cc_per_sensor_table(
    'relative_wasserstein_value', 'W1 (value)',
    r"W1 (value), scale-free (relative to the sensor's real per-case-mean median) --- is the "
    r"distribution of magnitudes right, as a fraction of the sensor's typical real scale."
)


% === process_1: 6 sensors x 20 mode/approach cols -- W1 (value) ===
\begin{table}[H]
\caption{Complete-profile evaluation for process\_1, experiment 951 (test evaluation). W1 (value), scale-free (relative to the sensor's real per-case-mean median) --- is the distribution of magnitudes right, as a fraction of the sensor's typical real scale. Lower is better. \textbf{Bold} = best mode/approach per sensor.}
\label{tab:complete\_profile\_relative\_wasserstein\_value\_951\_test\_process\_1}
\begin{tabular}{l|c|c|c|c|c|c|c|c|c|c|c|c|c|c|c|c|c|c|c|c}
\toprule
 & budget / baseline (Best) & budget / baseline (Baseline) & budget / ml\_global (Best) & budget / ml\_global (Baseline) & budget / ml\_local (Best) & budget / ml\_local (Baseline) & heuristic / baseline (Best) & heuristic / baseline (Baseline) & heuristic / ml\_global (Best) & heuristic / ml\_global (Baseline) & heuristic / ml\_local (Best) & heuristic / ml\_local (Baseline) & wip\_aware / baseline (Best) & wip\_aware / baseline (Basel

mode_label                                         budget / baseline         \
approach_label                                              Baseline   Best   
process   sensor_short                                                        
process_1 autoclave_cooling_water_demand_kW                    1.047  0.904   
          autoclave_steam_demand_kW                            1.369  1.415   
          bottling_power_kW                                    0.110  0.119   
          destillation_cooling_demand_kW                       4.257  4.301   
          destillation_steam_demand_kW                         6.017  6.068   
...                                                              ...    ...   
process_5 temp_nach_Kuehlturmkuehler_(WT6)_5s                  0.060  0.054   
          temp_nach_WR2,_vor_Druckerhoehungspumpe…             0.009  0.012   
          temp_nach_WR2_(WT2)_5s                               0.073  0.068   
          temp_vor_Vorwärmer_(WT_2)_1h                         0.047  0.033   
          vor_Vorwärmer_(WT_2)_5s                              0.023  0.059   

mode_label                                         budget / ml_global         \
approach_label                                               Baseline   Best   
process   sensor_short                                                         
process_1 autoclave_cooling_water_demand_kW                     1.023  0.756   
          autoclave_steam_demand_kW                             1.416  1.458   
          bottling_power_kW                                     0.100  0.114   
          destillation_cooling_demand_kW                        4.494  4.538   
          destillation_steam_demand_kW                          6.066  6.092   
...                                                               ...    ...   
process_5 temp_nach_Kuehlturmkuehler_(WT6)_5s                   0.060  0.051   
          temp_nach_WR2,_vor_Druckerhoehungspumpe…              0.012  0.013   
          temp_nach_WR2_(WT2)_5s                                0.058  0.066   
          temp_vor_Vorwärmer_(WT_2)_1h                          0.043  0.043   
          vor_Vorwärmer_(WT_2)_5s                               0.029  0.071   

mode_label                                         budget / ml_local         \
approach_label                                              Baseline   Best   
process   sensor_short                                                        
process_1 autoclave_cooling_water_demand_kW                    0.932  0.768   
          autoclave_steam_demand_kW                            1.396  1.455   
          bottling_power_kW                                    0.101  0.129   
          destillation_cooling_demand_kW                       4.352  4.386   
          destillation_steam_demand_kW                         5.964  5.981   
...                                                              ...    ...   
process_5 temp_nach_Kuehlturmkuehler_(WT6)_5s                  0.061  0.060   
          temp_nach_WR2,_vor_Druckerhoehungspumpe…             0.009  0.012   
          temp_nach_WR2_(WT2)_5s                               0.062  0.065   
          temp_vor_Vorwärmer_(WT_2)_1h                         0.043  0.041   
          vor_Vorwärmer_(WT_2)_5s                              0.024  0.071   

mode_label                                         heuristic / baseline  \
approach_label                                                 Baseline   
process   sensor_short                                                    
process_1 autoclave_cooling_water_demand_kW                       0.949   
          autoclave_steam_demand_kW                               1.243   
          bottling_power_kW                                       0.110   
          destillation_cooling_demand_kW                          2.765   
          destillation_steam_demand_kW                            3.712   
...                                                              

### Aggregated ranking (complete-profile, one number per process x method)

Same rollup as the Energy-Distribution ranking above, applied to the complete-profile
metrics: median over sensors per process, then median over processes for `Overall`.
Each row is one (mode, curve approach) combination. Rows sorted best-to-worst.


In [10]:
# -- Aggregated ranking: complete-profile W1 (time) / W1 (value), relative --
def _cc_ranking_table(metric_col, metric_label):
    if df_complete_curve.empty:
        print('Nothing to show -- df_complete_curve is empty.')
        return None
    dfc = df_complete_curve[df_complete_curve['approach'].isin(_cc_approach_labels)].copy()
    dfc['mode_label'] = dfc['mode'].map(display_mode)
    dfc['approach_label'] = dfc['approach'].map(_cc_approach_labels)
    dfc['method_label'] = dfc['mode_label'] + ' (' + dfc['approach_label'] + ')'

    rank_agg = dfc.groupby(['process', 'method_label'])[metric_col].median().reset_index()
    rank_pivot = rank_agg.pivot(index='method_label', columns='process', values=metric_col)
    rank_pivot['Overall'] = rank_pivot.median(axis=1)
    rank_pivot = rank_pivot.sort_values('Overall')

    str_table = pd.DataFrame(index=rank_pivot.index, columns=rank_pivot.columns, dtype=object)
    for col in rank_pivot.columns:
        vals = rank_pivot[col].dropna()
        if vals.empty:
            str_table[col] = ''
            continue
        best_val = vals.min()
        for idx in rank_pivot.index:
            val = rank_pivot.loc[idx, col]
            if pd.isna(val):
                str_table.loc[idx, col] = ''
                continue
            fmt = f'{val:.3f}'
            str_table.loc[idx, col] = (r'\textbf{' + fmt + r'}') if abs(val - best_val) < 1e-6 else fmt

    n_cols = len(rank_pivot.columns)
    col_format = 'l|' + '|'.join(['c'] * (n_cols - 1)) + '|c'
    rank_latex = str_table.to_latex(
        escape=False, column_format=col_format,
        caption=(
            f'Aggregated complete-profile ranking, experiment {EXPERIMENT} ({SPLIT} evaluation), {metric_label}. '
            r'Median relative distance over sensors per process, then median over processes for Overall. '
            r'Lower is better. Rows sorted best-to-worst by Overall. \textbf{Bold} = best method per column.'
        ),
        label=f'tab:complete_profile_ranking_{metric_col}_{EXPERIMENT}_{SPLIT}',
        position='H',
    )
    print(rank_latex.replace('_', r'\_'))
    display(rank_pivot.round(3))
    return rank_pivot

_cc_time_rank = _cc_ranking_table('relative_wasserstein_time', 'W1 (time)')


\begin{table}[H]
\caption{Aggregated complete-profile ranking, experiment 951 (test evaluation), W1 (time). Median relative distance over sensors per process, then median over processes for Overall. Lower is better. Rows sorted best-to-worst by Overall. \textbf{Bold} = best method per column.}
\label{tab:complete\_profile\_ranking\_relative\_wasserstein\_time\_951\_test}
\begin{tabular}{l|c|c|c|c|c|c|c}
\toprule
process & process\_1 & process\_2 & process\_3 & process\_4\_1 & process\_4\_2 & process\_5 & Overall \\
method\_label &  &  &  &  &  &  &  \\
\midrule
budget / ml\_local (Best) & \textbf{0.020} & \textbf{0.151} & 0.242 & 0.202 & 0.195 & 0.270 & \textbf{0.199} \\
budget / ml\_local (Baseline) & 0.020 & 0.152 & 0.241 & 0.203 & 0.197 & 0.270 & 0.200 \\
budget / baseline (Best) & 0.096 & 0.247 & \textbf{0.189} & 0.202 & 0.235 & \textbf{0.203} & 0.203 \\
budget / baseline (Baseline) & 0.091 & 0.245 & 0.198 & 0.206 & 0.235 & 0.205 & 0.205 \\
heuristic / ml\_local (Best) & 0.023 & 0.

process,process_1,process_2,process_3,process_4_1,process_4_2,process_5,Overall
method_label,,,,,,,
budget / ml_local (Best),0.020,0.151,0.242,0.202,0.195,0.270,0.199
budget / ml_local (Baseline),0.020,0.152,0.241,0.203,0.197,0.270,0.200
budget / baseline (Best),0.096,0.247,0.189,0.202,0.235,0.203,0.203
budget / baseline (Baseline),0.091,0.245,0.198,0.206,0.235,0.205,0.205
heuristic / ml_local (Best),0.023,0.187,0.312,0.204,0.213,0.451,0.209
heuristic / ml_local (Baseline),0.023,0.207,0.313,0.204,0.213,0.451,0.210
budget / ml_global (Best),0.072,0.176,0.222,0.214,0.210,0.359,0.212
budget / ml_global (Baseline),0.071,0.176,0.229,0.215,0.212,0.359,0.214
wip_aware / ml_global (Baseline),0.077,2.018,0.240,0.194,0.151,0.466,0.217


In [11]:
_cc_value_rank = _cc_ranking_table('relative_wasserstein_value', 'W1 (value)')


\begin{table}[H]
\caption{Aggregated complete-profile ranking, experiment 951 (test evaluation), W1 (value). Median relative distance over sensors per process, then median over processes for Overall. Lower is better. Rows sorted best-to-worst by Overall. \textbf{Bold} = best method per column.}
\label{tab:complete\_profile\_ranking\_relative\_wasserstein\_value\_951\_test}
\begin{tabular}{l|c|c|c|c|c|c|c}
\toprule
process & process\_1 & process\_2 & process\_3 & process\_4\_1 & process\_4\_2 & process\_5 & Overall \\
method\_label &  &  &  &  &  &  &  \\
\midrule
budget / baseline (Baseline) & 1.208 & 0.121 & 0.141 & 0.112 & 0.256 & 0.135 & \textbf{0.138} \\
budget / ml\_local (Baseline) & 1.164 & \textbf{0.110} & 0.148 & 0.092 & 0.272 & 0.131 & 0.140 \\
wip\_aware / ml\_global (Baseline) & 1.236 & 0.132 & 0.148 & 0.092 & 0.257 & 0.122 & 0.140 \\
heuristic / ml\_local (Baseline) & 1.168 & 0.147 & \textbf{0.140} & 0.091 & 0.259 & 0.129 & 0.143 \\
heuristic / ml\_global (Baseline) & 1.22

process,process_1,process_2,process_3,process_4_1,process_4_2,process_5,Overall
method_label,,,,,,,
budget / baseline (Baseline),1.208,0.121,0.141,0.112,0.256,0.135,0.138
budget / ml_local (Baseline),1.164,0.110,0.148,0.092,0.272,0.131,0.140
wip_aware / ml_global (Baseline),1.236,0.132,0.148,0.092,0.257,0.122,0.140
heuristic / ml_local (Baseline),1.168,0.147,0.140,0.091,0.259,0.129,0.143
heuristic / ml_global (Baseline),1.227,0.139,0.148,0.094,0.257,0.122,0.144
wip_aware / ml_local (Baseline),1.169,0.149,0.140,0.092,0.257,0.129,0.144
budget / ml_global (Baseline),1.219,0.124,0.168,0.091,0.258,0.130,0.149
heuristic / baseline (Baseline),1.096,0.202,0.159,0.110,0.258,0.123,0.181
wip_aware / ml_local (Best),1.007,0.147,0.156,0.334,0.222,0.133,0.189


### Combined ranking (W1 time + W1 shape pooled into one score)

The two rankings above keep timing and magnitude separate. This one pools both into a
**single** median per method -- valid only because both are already the scale-free
`relative_wasserstein_*` versions (unitless: divided by real median case duration / real
per-case-mean median respectively), never the raw `wasserstein_time_median` (minutes) /
`wasserstein_value_median` (raw sensor units), which live on completely different scales
and could never be pooled directly. Median over (sensor x metric) per process, then
median over processes for `Overall`. Rows sorted best-to-worst.


In [12]:
# -- Combined complete-profile ranking: W1 (time) + W1 (value), pooled into one score --
def _cc_combined_ranking_table():
    if df_complete_curve.empty:
        print('Nothing to show -- df_complete_curve is empty.')
        return None
    dfc = df_complete_curve[df_complete_curve['approach'].isin(_cc_approach_labels)].copy()
    dfc['mode_label'] = dfc['mode'].map(display_mode)
    dfc['approach_label'] = dfc['approach'].map(_cc_approach_labels)
    dfc['method_label'] = dfc['mode_label'] + ' (' + dfc['approach_label'] + ')'

    # Stack both scale-free metrics into one pooled column -- each row becomes
    # two rows (one per metric), then a single median combines "is timing
    # right" and "is magnitude right" into one standardized score.
    _long = pd.concat([
        dfc[['process', 'method_label', 'relative_wasserstein_time']]
            .rename(columns={'relative_wasserstein_time': 'value'}),
        dfc[['process', 'method_label', 'relative_wasserstein_value']]
            .rename(columns={'relative_wasserstein_value': 'value'}),
    ], ignore_index=True)

    rank_agg = _long.groupby(['process', 'method_label'])['value'].median().reset_index()
    rank_pivot = rank_agg.pivot(index='method_label', columns='process', values='value')
    rank_pivot['Overall'] = rank_pivot.median(axis=1)
    rank_pivot = rank_pivot.sort_values('Overall')

    str_table = pd.DataFrame(index=rank_pivot.index, columns=rank_pivot.columns, dtype=object)
    for col in rank_pivot.columns:
        vals = rank_pivot[col].dropna()
        if vals.empty:
            str_table[col] = ''
            continue
        best_val = vals.min()
        for idx in rank_pivot.index:
            val = rank_pivot.loc[idx, col]
            if pd.isna(val):
                str_table.loc[idx, col] = ''
                continue
            fmt = f'{val:.3f}'
            str_table.loc[idx, col] = (r'\textbf{' + fmt + r'}') if abs(val - best_val) < 1e-6 else fmt

    n_cols = len(rank_pivot.columns)
    col_format = 'l|' + '|'.join(['c'] * (n_cols - 1)) + '|c'
    rank_latex = str_table.to_latex(
        escape=False, column_format=col_format,
        caption=(
            f'Aggregated complete-profile ranking, experiment {EXPERIMENT} ({SPLIT} evaluation), '
            r'W1 (time) and W1 (value) pooled into one standardized score. '
            r'Median relative distance over sensors and both metrics per process, '
            r'then median over processes for Overall. '
            r'Lower is better. Rows sorted best-to-worst by Overall. \textbf{Bold} = best method per column.'
        ),
        label=f'tab:complete_profile_ranking_combined_{EXPERIMENT}_{SPLIT}',
        position='H',
    )
    print(rank_latex.replace('_', r'\_'))
    display(rank_pivot.round(3))
    return rank_pivot

_cc_combined_rank = _cc_combined_ranking_table()


\begin{table}[H]
\caption{Aggregated complete-profile ranking, experiment 951 (test evaluation), W1 (time) and W1 (value) pooled into one standardized score. Median relative distance over sensors and both metrics per process, then median over processes for Overall. Lower is better. Rows sorted best-to-worst by Overall. \textbf{Bold} = best method per column.}
\label{tab:complete\_profile\_ranking\_combined\_951\_test}
\begin{tabular}{l|c|c|c|c|c|c|c}
\toprule
process & process\_1 & process\_2 & process\_3 & process\_4\_1 & process\_4\_2 & process\_5 & Overall \\
method\_label &  &  &  &  &  &  &  \\
\midrule
budget / baseline (Baseline) & 0.338 & 0.185 & 0.187 & \textbf{0.183} & 0.238 & 0.203 & \textbf{0.195} \\
wip\_aware / ml\_global (Baseline) & 0.160 & 0.772 & 0.211 & 0.193 & 0.151 & 0.424 & 0.202 \\
heuristic / ml\_local (Baseline) & 0.131 & 0.172 & 0.252 & 0.203 & 0.213 & 0.412 & 0.208 \\
wip\_aware / ml\_local (Baseline) & \textbf{0.130} & 0.784 & 0.226 & 0.192 & 0.149 & 0.412 &

process,process_1,process_2,process_3,process_4_1,process_4_2,process_5,Overall
method_label,,,,,,,
budget / baseline (Baseline),0.338,0.185,0.187,0.183,0.238,0.203,0.195
wip_aware / ml_global (Baseline),0.160,0.772,0.211,0.193,0.151,0.424,0.202
heuristic / ml_local (Baseline),0.131,0.172,0.252,0.203,0.213,0.412,0.208
wip_aware / ml_local (Baseline),0.130,0.784,0.226,0.192,0.149,0.412,0.209
heuristic / ml_local (Best),0.142,0.176,0.269,0.204,0.214,0.442,0.209
budget / ml_local (Baseline),0.329,0.150,0.217,0.201,0.199,0.268,0.209
budget / baseline (Best),0.276,0.208,0.183,0.213,0.233,0.202,0.210
budget / ml_global (Baseline),0.355,0.153,0.205,0.211,0.212,0.348,0.212
heuristic / ml_global (Best),0.144,0.198,0.260,0.207,0.222,0.462,0.215


# Schedule Profile Evaluation -- Direct Method vs. Proposed (Process-Simulation) Approach

Per process: compares three ways to get from "a case is about to run" to "predicted
complete energy profile", all evaluated against the exact same real test cases:

1. **Schedule-direct** -- a case-level regression straight from schedule-only features
   (recipe/case attributes, start time, external factors), no process simulation at all.
2. **Best, mine** -- the full proposed pipeline: process simulation (best-fidelity mode
   for that process) + DTW + Ext. Factors + Prev. Activity curve prediction.
3. **Stochastic generator** -- a per-position Normal fit with no conditioning at all, the
   floor everything else needs to beat.

Source: `schedule_profile_eval_results/<process>/schedule_profile_eval_summary.csv`
(requires `run_schedule_profile_eval: True` and `run_energy_modelling: True` in
`pipeline.py`). The raw `*_wasserstein_time` (minutes) / `*_wasserstein_value` (raw
sensor units) columns aren't comparable across sensors of different units, so -- same
standardization as the Complete-Curve section above -- each is divided by a real,
model-independent scale reference before aggregating:

- `W1 (time), rel.` = `*_wasserstein_time` / real median case duration for that process.
- `W1 (value), rel.` = `*_wasserstein_value` / real per-case-mean median for that sensor.
- `Combined` = both standardized metrics pooled together, one median -- valid only
  because both are already unitless.


In [13]:
# -- Standardize + aggregate the Schedule Profile Evaluation, per process --
# Per-case normalization: each case is normalized by its OWN real curve's
# scale (own real duration for time; own real mean/std -- z-score -- for
# value) BEFORE computing the Wasserstein distance, mirroring the per-
# instance sMAE normalization used in the individual energy-profile
# evaluation (evaluate_pipeline_on_test). This replaces the earlier version,
# which divided an aggregated (already medianed) raw W1 by one global
# process-wide / sensor-wide reference constant -- that's mathematically
# identical to normalizing-then-aggregating ONLY when every case shares the
# same reference value, which isn't true here: real case durations and real
# per-case magnitudes vary quite a bit. A single global constant lets cases
# with an unusually long/short real duration (or an unusually flat/active
# real curve) get over- or under-penalized purely because of their own
# scale, not prediction quality.
#
# Degenerate cases are excluded case-by-case (NaN) rather than blowing up the
# ratio -- mirroring the `_cv_ok` guard used for sMAE:
#   - time:  skip if this case's own real duration is ~0
#   - value: skip if this case's own real curve is ~constant (std ~0, or
#            coefficient of variation <= 1%) -- z-scoring a flat signal turns
#            noise-level wiggles into huge, meaningless z-scores.
_sp_series = {
    'Schedule-direct':          'schedule',
    'Best, mine':               'best',
    'Best, duration-corrected': 'best_duration_corrected',
    'Stochastic generator':     'stochastic',
    'Stochastic (bootstrap)':   'bootstrap',
}

def _load_schedule_profile_cases(proc):
    from scipy.stats import wasserstein_distance
    """Returns a long DataFrame [process, sensor, method, case_id, w1_time_rel,
    w1_value_rel] -- one row per individual case instance, no aggregation at
    all -- for one process, or an empty DataFrame if the data isn't available."""
    curves_path = run_dir / 'schedule_profile_eval_results' / proc / 'predicted_curves.parquet'
    if not curves_path.exists():
        return pd.DataFrame()
    df_curves = pd.read_parquet(curves_path)

    case_rows = []
    for sensor, sensor_g in df_curves.groupby('sensor'):
        real_by_case = {cid: g.sort_values('t_minutes')
                         for cid, g in sensor_g[sensor_g['series'] == 'real'].groupby('case_id')}
        for method, series_name in _sp_series.items():
            pred_by_case = {cid: g.sort_values('t_minutes')
                             for cid, g in sensor_g[sensor_g['series'] == series_name].groupby('case_id')}
            for case_id, real_g in real_by_case.items():
                pred_g = pred_by_case.get(case_id)
                if pred_g is None or pred_g.empty:
                    continue
                t_real = real_g['t_minutes'].to_numpy()
                v_real = real_g['value'].to_numpy()
                t_pred = pred_g['t_minutes'].to_numpy()
                v_pred = pred_g['value'].to_numpy()

                w_real = np.clip(v_real, 0, None)
                w_pred = np.clip(v_pred, 0, None)
                if w_real.sum() <= 0 or w_pred.sum() <= 0:
                    continue

                # -- Time: normalize by this case's own real duration --
                duration_real = float(t_real.max())
                t_rel = float('nan')
                if duration_real > 1e-6:
                    t_rel = float(wasserstein_distance(
                        t_real / duration_real, t_pred / duration_real,
                        u_weights=w_real, v_weights=w_pred))

                # -- Value: z-score by this case's own real mean/std --
                mu_real, sig_real = float(v_real.mean()), float(v_real.std())
                _cv_ok = sig_real > 1e-10 and (mu_real == 0 or (sig_real / abs(mu_real)) > 0.01)
                v_rel = float('nan')
                if _cv_ok:
                    z_real = (v_real - mu_real) / sig_real
                    z_pred = (v_pred - mu_real) / sig_real
                    v_rel = float(wasserstein_distance(z_real, z_pred))

                case_rows.append({'process': proc, 'sensor': sensor, 'method': method,
                                  'case_id': case_id, 'w1_time_rel': t_rel, 'w1_value_rel': v_rel})

    return pd.DataFrame(case_rows)


def _load_schedule_profile_relative(proc):
    """Returns a long DataFrame [process, sensor, method, w1_time_rel, w1_value_rel]
    for one process, or an empty DataFrame if the data isn't available. Median
    across cases, per (process, sensor, method) -- NaN (degenerate/excluded)
    cases are dropped automatically by .median()."""
    df_cases = _load_schedule_profile_cases(proc)
    if df_cases.empty:
        return pd.DataFrame()
    return df_cases.groupby(['process', 'sensor', 'method']).agg(
        w1_time_rel=('w1_time_rel', 'median'),
        w1_value_rel=('w1_value_rel', 'median'),
    ).reset_index()


def _bold_best_per_column(agg):
    str_table = agg.round(3).astype(object)
    for col in agg.columns:
        vals = agg[col].dropna()
        if vals.empty:
            continue
        best_val = vals.min()
        for idx in agg.index:
            val = agg.loc[idx, col]
            if pd.isna(val):
                str_table.loc[idx, col] = ''
                continue
            fmt = f'{val:.3f}'
            str_table.loc[idx, col] = (r'\textbf{' + fmt + r'}') if abs(val - best_val) < 1e-6 else fmt
    return str_table


_sp_all_processes = [
    p for p in sorted({d.name for d in (run_dir / 'schedule_profile_eval_results').iterdir()})
] if (run_dir / 'schedule_profile_eval_results').exists() else []
print(f'Processes with Schedule Profile Evaluation data: {_sp_all_processes}')

df_sp_relative_all = pd.concat(
    [_load_schedule_profile_relative(p) for p in _sp_all_processes], ignore_index=True
) if _sp_all_processes else pd.DataFrame()

# Fair comparison: only keep (process, sensor) pairs where ALL methods produced a
# valid (non-NaN) w1_time_rel and w1_value_rel. Otherwise a method that's missing a
# sensor entirely (e.g. no trained pipeline for it) gets a smaller, easier pool of
# sensors than the others, which silently biases its median up or down.
if not df_sp_relative_all.empty:
    _n_methods_expected = len(_sp_series)
    _complete_counts = (
        df_sp_relative_all.dropna(subset=['w1_time_rel', 'w1_value_rel'])
        .groupby(['process', 'sensor']).size()
    )
    _complete_keys = set(_complete_counts[_complete_counts == _n_methods_expected].index)
    _all_keys = set(zip(df_sp_relative_all['process'], df_sp_relative_all['sensor']))
    _dropped_keys = sorted(_all_keys - _complete_keys)
    if _dropped_keys:
        print(f'[Schedule Profile Eval] Dropping {len(_dropped_keys)} (process, sensor) pair(s) '
              f'lacking data for all {_n_methods_expected} methods, so every method is aggregated '
              f'over the same sensors:')
        for _proc_k, _sensor_k in _dropped_keys:
            print(f'    {_proc_k} / {_sensor_k}')
    df_sp_relative_all = df_sp_relative_all[
        df_sp_relative_all.set_index(['process', 'sensor']).index.isin(_complete_keys)
    ].reset_index(drop=True)

# Raw, case-level rows (no per-sensor or per-process aggregation at all),
# restricted to the same fair (process, sensor) set as above -- feeds the
# single flat median table below.
df_sp_cases_all = pd.concat(
    [_load_schedule_profile_cases(p) for p in _sp_all_processes], ignore_index=True
) if _sp_all_processes else pd.DataFrame()
if not df_sp_cases_all.empty:
    df_sp_cases_all = df_sp_cases_all[
        df_sp_cases_all.set_index(['process', 'sensor']).index.isin(_complete_keys)
    ].reset_index(drop=True)

for proc in _sp_all_processes:
    sub = df_sp_relative_all[df_sp_relative_all['process'] == proc]
    if sub.empty:
        print(f'{proc}: no data, skipping.')
        continue

    agg = sub.groupby('method').agg(
        **{'W1 (time), rel. median': ('w1_time_rel', 'median'),
           'W1 (value), rel. median': ('w1_value_rel', 'median')}
    )
    # Combined: give the two axes EQUAL weight. Raw-pooling w1_time (~0.1)
    # with w1_value (~1.4) lets value dominate ~10:1 and hides the timing
    # result, so instead express each axis relative to the stochastic floor
    # (the baseline every method must beat) and average -- both are then
    # O(1) and equally weighted. <1 == better than the stochastic floor.
    _floor_t = agg.loc['Stochastic generator', 'W1 (time), rel. median'] if 'Stochastic generator' in agg.index else np.nan
    _floor_v = agg.loc['Stochastic generator', 'W1 (value), rel. median'] if 'Stochastic generator' in agg.index else np.nan
    _rel_t = agg['W1 (time), rel. median'] / _floor_t if (_floor_t and _floor_t > 0) else agg['W1 (time), rel. median']
    _rel_v = agg['W1 (value), rel. median'] / _floor_v if (_floor_v and _floor_v > 0) else agg['W1 (value), rel. median']
    agg['Combined (floor-rel.)'] = 0.5 * (_rel_t + _rel_v)
    agg = agg.sort_values('Combined (floor-rel.)')

    str_table = _bold_best_per_column(agg)
    latex = str_table.to_latex(
        escape=False, column_format='l|c|c|c',
        caption=(
            f'Aggregated Schedule Profile Evaluation for {proc}, experiment {EXPERIMENT} ({SPLIT} evaluation). '
            r'One standardized (scale-free) number per method, median over sensors '
            r'(Combined: each axis relative to the stochastic floor, then averaged with equal weight -- <1 beats the floor). '
            r'Lower is better. Rows sorted best-to-worst by Combined. \textbf{Bold} = best method per column.'
        ),
        label=f'tab:schedule_profile_agg_{EXPERIMENT}_{SPLIT}_{proc}',
        position='H',
    )
    print(f'% === {proc} ===')
    print(latex.replace('_', r'\_'))
    display(agg.round(3))
    print()


Processes with Schedule Profile Evaluation data: ['process_1', 'process_2', 'process_3', 'process_4_1', 'process_4_2', 'process_5']
[Schedule Profile Eval] Dropping 2 (process, sensor) pair(s) lacking data for all 5 methods, so every method is aggregated over the same sensors:
    process_1 / water_supply_power_kW_energy_to_model
    process_4_2 / (14)_filter_mas_kg/h_energy_to_model
% === process_1 ===
\begin{table}[H]
\caption{Aggregated Schedule Profile Evaluation for process\_1, experiment 951 (test evaluation). One standardized (scale-free) number per method, median over sensors (Combined: each axis relative to the stochastic floor, then averaged with equal weight -- <1 beats the floor). Lower is better. Rows sorted best-to-worst by Combined. \textbf{Bold} = best method per column.}
\label{tab:schedule\_profile\_agg\_951\_test\_process\_1}
\begin{tabular}{l|c|c|c}
\toprule
 & W1 (time), rel. median & W1 (value), rel. median & Combined (floor-rel.) \\
method &  &  &  \\
\midrule
Be

,"W1 (time), rel. median","W1 (value), rel. median",Combined (floor-rel.)
method,,,
"Best, mine",0.031,0.232,0.610
Schedule-direct,0.088,0.133,0.637
Stochastic (bootstrap),0.112,0.122,0.711
"Best, duration-corrected",0.072,0.232,0.778
Stochastic generator,0.123,0.239,1.000



% === process_2 ===
\begin{table}[H]
\caption{Aggregated Schedule Profile Evaluation for process\_2, experiment 951 (test evaluation). One standardized (scale-free) number per method, median over sensors (Combined: each axis relative to the stochastic floor, then averaged with equal weight -- <1 beats the floor). Lower is better. Rows sorted best-to-worst by Combined. \textbf{Bold} = best method per column.}
\label{tab:schedule\_profile\_agg\_951\_test\_process\_2}
\begin{tabular}{l|c|c|c}
\toprule
 & W1 (time), rel. median & W1 (value), rel. median & Combined (floor-rel.) \\
method &  &  &  \\
\midrule
Stochastic (bootstrap) & 0.176 & \textbf{0.140} & \textbf{0.674} \\
Best, duration-corrected & 0.193 & 0.142 & 0.725 \\
Schedule-direct & \textbf{0.162} & 0.248 & 0.773 \\
Best, mine & 0.219 & 0.142 & 0.798 \\
Stochastic generator & 0.179 & 0.387 & 1.000 \\
\bottomrule
\end{tabular}
\end{table}



,"W1 (time), rel. median","W1 (value), rel. median",Combined (floor-rel.)
method,,,
Stochastic (bootstrap),0.176,0.140,0.674
"Best, duration-corrected",0.193,0.142,0.725
Schedule-direct,0.162,0.248,0.773
"Best, mine",0.219,0.142,0.798
Stochastic generator,0.179,0.387,1.000



% === process_3 ===
\begin{table}[H]
\caption{Aggregated Schedule Profile Evaluation for process\_3, experiment 951 (test evaluation). One standardized (scale-free) number per method, median over sensors (Combined: each axis relative to the stochastic floor, then averaged with equal weight -- <1 beats the floor). Lower is better. Rows sorted best-to-worst by Combined. \textbf{Bold} = best method per column.}
\label{tab:schedule\_profile\_agg\_951\_test\_process\_3}
\begin{tabular}{l|c|c|c}
\toprule
 & W1 (time), rel. median & W1 (value), rel. median & Combined (floor-rel.) \\
method &  &  &  \\
\midrule
Schedule-direct & 0.186 & 0.211 & \textbf{0.831} \\
Stochastic (bootstrap) & 0.193 & \textbf{0.200} & 0.843 \\
Best, duration-corrected & 0.216 & 0.236 & 0.956 \\
Stochastic generator & \textbf{0.153} & 0.472 & 1.000 \\
Best, mine & 0.238 & 0.236 & 1.028 \\
\bottomrule
\end{tabular}
\end{table}



...

,"W1 (time), rel. median","W1 (value), rel. median",Combined (floor-rel.)
method,,,
Schedule-direct,0.019,1.301,0.924
Stochastic (bootstrap),0.020,1.413,0.983
Stochastic generator,0.019,1.587,1.000
"Best, duration-corrected",0.053,1.134,1.769
"Best, mine",0.206,1.134,5.848



% === process_4_2 ===
\begin{table}[H]
\caption{Aggregated Schedule Profile Evaluation for process\_4\_2, experiment 951 (test evaluation). One standardized (scale-free) number per method, median over sensors (Combined: each axis relative to the stochastic floor, then averaged with equal weight -- <1 beats the floor). Lower is better. Rows sorted best-to-worst by Combined. \textbf{Bold} = best method per column.}
\label{tab:schedule\_profile\_agg\_951\_test\_process\_4\_2}
\begin{tabular}{l|c|c|c}
\toprule
 & W1 (time), rel. median & W1 (value), rel. median & Combined (floor-rel.) \\
method &  &  &  \\
\midrule
Schedule-direct & \textbf{0.016} & 2.032 & \textbf{0.839} \\
Stochastic (bootstrap) & 0.018 & \textbf{1.909} & 0.903 \\
Stochastic generator & 0.016 & 2.796 & 1.000 \\
Best, duration-corrected & 0.037 & 2.392 & 1.547 \\
Best, mine & 0.192 & 2.392 & 6.260 \\
\bottomrule
\end{tabular}
\end{table}



,"W1 (time), rel. median","W1 (value), rel. median",Combined (floor-rel.)
method,,,
Schedule-direct,0.016,2.032,0.839
Stochastic (bootstrap),0.018,1.909,0.903
Stochastic generator,0.016,2.796,1.000
"Best, duration-corrected",0.037,2.392,1.547
"Best, mine",0.192,2.392,6.260



% === process_5 ===
\begin{table}[H]
\caption{Aggregated Schedule Profile Evaluation for process\_5, experiment 951 (test evaluation). One standardized (scale-free) number per method, median over sensors (Combined: each axis relative to the stochastic floor, then averaged with equal weight -- <1 beats the floor). Lower is better. Rows sorted best-to-worst by Combined. \textbf{Bold} = best method per column.}
\label{tab:schedule\_profile\_agg\_951\_test\_process\_5}
\begin{tabular}{l|c|c|c}
\toprule
 & W1 (time), rel. median & W1 (value), rel. median & Combined (floor-rel.) \\
method &  &  &  \\
\midrule
Schedule-direct & 0.062 & \textbf{1.568} & \textbf{0.803} \\
Stochastic (bootstrap) & \textbf{0.062} & 2.031 & 0.892 \\
Best, duration-corrected & 0.065 & 2.189 & 0.950 \\
Stochastic generator & 0.063 & 2.534 & 1.000 \\
Best, mine & 0.307 & 2.189 & 2.880 \\
\bottomrule
\end{tabular}
\end{table}



,"W1 (time), rel. median","W1 (value), rel. median",Combined (floor-rel.)
method,,,
Schedule-direct,0.062,1.568,0.803
Stochastic (bootstrap),0.062,2.031,0.892
"Best, duration-corrected",0.065,2.189,0.950
Stochastic generator,0.063,2.534,1.000
"Best, mine",0.307,2.189,2.880


## Population-Level (Unpaired) Distributional Evaluation

The Schedule Profile Evaluation above pairs each method's prediction for a case with that SAME real case's curve -- the right test for a discriminative, case-conditioned predictor (schedule-direct), but the wrong test for a stochastic/generative method (the process simulation, the per-position stochastic generator, the empirical bootstrap): those methods aren't trying to reproduce any one specific real case, they're trying to produce a realistic POPULATION of cases. Scoring a generative method by how close one random draw lands to one specific real curve structurally punishes it for not being a point predictor.

This section instead pools ALL real cases and ALL of a method's cases (no case_id pairing at all, strictly within one process/sensor at a time -- never pooled across sensors or processes) and asks how different those two populations are, on two sensor-type-agnostic axes:

- **W1 (shape)** -- population-level generalization of the per-case W1-time metric above: pools every (relative-time, value) point from ALL real cases into one value-weighted distribution over relative time (0..1), does the same for the method's cases, and computes ONE Wasserstein distance between the two pooled distributions. Already scale-free (0..1 axis).
- **W1 (magnitude)** -- pools every raw sensor reading (no per-case reduction at all) and compares the pooled real vs. method value distributions directly, via `compare_pooled_value_distributions` -- valid for any sensor type, including intensive quantities like temperature where a per-case sum/total would be meaningless.

See `compare_population_shape` and `compare_pooled_value_distributions` in sim_extractor.py.

In [14]:
# -- Population-Level (unpaired) Distributional Evaluation, per process --
# Same scale-free-relative / floor-normalized / bold-best-per-column conventions
# as the per-case Schedule Profile Evaluation above, reusing _sp_series (so
# 'Best, budget' stays excluded here too -- it's the per-process overall_error
# winner's own family and would just duplicate 'Best, mine' wherever it wins).

def _load_population_eval(proc):
    """Returns a DataFrame [process, sensor, method, w1_shape_rel,
    w1_magnitude_rel] -- one row per (sensor, method), already relative to
    the REAL population's own scale for that sensor -- or an empty
    DataFrame if the data isn't available for this process."""
    pop_path = run_dir / 'schedule_profile_eval_results' / proc / 'population_distribution_eval.csv'
    if not pop_path.exists():
        return pd.DataFrame()
    df_pop = pd.read_csv(pop_path)

    rows = []
    for method, series_name in _sp_series.items():
        sub = df_pop[df_pop['method'] == series_name]
        for _, r in sub.iterrows():
            shape_rel = (r['w1_shape'] / r['real_shape_scale']
                         if r['real_shape_scale'] > 1e-9 else float('nan'))
            mag_rel = (r['w1_magnitude'] / r['real_magnitude_scale']
                       if r['real_magnitude_scale'] > 1e-9 else float('nan'))
            rows.append({'process': proc, 'sensor': r['sensor'], 'method': method,
                        'w1_shape_rel': shape_rel, 'w1_magnitude_rel': mag_rel})
    return pd.DataFrame(rows)


df_pop_relative_all = pd.concat(
    [_load_population_eval(p) for p in _sp_all_processes], ignore_index=True
) if _sp_all_processes else pd.DataFrame()

# Same fairness guard as the per-case table: only keep (process, sensor) pairs
# where every method produced a valid number for both axes.
if not df_pop_relative_all.empty:
    _n_methods_expected_pop = len(_sp_series)
    _complete_counts_pop = (
        df_pop_relative_all.dropna(subset=['w1_shape_rel', 'w1_magnitude_rel'])
        .groupby(['process', 'sensor']).size()
    )
    _complete_keys_pop = set(_complete_counts_pop[_complete_counts_pop == _n_methods_expected_pop].index)
    _all_keys_pop = set(zip(df_pop_relative_all['process'], df_pop_relative_all['sensor']))
    _dropped_keys_pop = sorted(_all_keys_pop - _complete_keys_pop)
    if _dropped_keys_pop:
        print(f'[Population Eval] Dropping {len(_dropped_keys_pop)} (process, sensor) pair(s) '
              f'lacking data for all {_n_methods_expected_pop} methods:')
        for _proc_k, _sensor_k in _dropped_keys_pop:
            print(f'    {_proc_k} / {_sensor_k}')
    df_pop_relative_all = df_pop_relative_all[
        df_pop_relative_all.set_index(['process', 'sensor']).index.isin(_complete_keys_pop)
    ].reset_index(drop=True)

for proc in _sp_all_processes:
    sub = df_pop_relative_all[df_pop_relative_all['process'] == proc] if not df_pop_relative_all.empty else pd.DataFrame()
    if sub.empty:
        print(f'{proc}: no population-eval data, skipping.')
        continue

    agg = sub.groupby('method').agg(
        **{'W1 (shape, population), rel. median':     ('w1_shape_rel', 'median'),
           'W1 (magnitude, population), rel. median': ('w1_magnitude_rel', 'median')}
    )
    agg = agg.sort_values('W1 (shape, population), rel. median')

    str_table = _bold_best_per_column(agg)
    latex = str_table.to_latex(
        escape=False, column_format='l|c|c',
        caption=(
            f'Aggregated Population-Level (unpaired) Distributional Evaluation for {proc}, '
            f'experiment {EXPERIMENT} ({SPLIT} evaluation). Real and method cases pooled per '
            r'sensor (no case-to-case pairing), median over sensors. Shape: population '
            r'generalization of the per-case W1-time metric (already scale-free). Magnitude: '
            r'pooled raw-value W1 (sensor-type-agnostic, works for temperature too), relative '
            r'to the real population\'s own value std. Lower is better. Rows sorted best-to-worst '
            r'by shape. \textbf{Bold} = best method per column.'
        ),
        label=f'tab:population_eval_agg_{EXPERIMENT}_{SPLIT}_{proc}',
        position='H',
    )
    print(f'% === {proc} ===')
    print(latex.replace('_', r'\_'))
    display(agg.round(3))
    print()


[Population Eval] Dropping 2 (process, sensor) pair(s) lacking data for all 5 methods:
    process_1 / water_supply_power_kW_energy_to_model
    process_4_2 / (14)_filter_mas_kg/h_energy_to_model
% === process_1 ===
\begin{table}[H]
\caption{Aggregated Population-Level (unpaired) Distributional Evaluation for process\_1, experiment 951 (test evaluation). Real and method cases pooled per sensor (no case-to-case pairing), median over sensors. Shape: population generalization of the per-case W1-time metric (already scale-free). Magnitude: pooled raw-value W1 (sensor-type-agnostic, works for temperature too), relative to the real population\'s own value std. Lower is better. Rows sorted best-to-worst by shape. \textbf{Bold} = best method per column.}
\label{tab:population\_eval\_agg\_951\_test\_process\_1}
\begin{tabular}{l|c|c}
\toprule
 & W1 (shape, population), rel. median & W1 (magnitude, population), rel. median \\
method &  &  \\
\midrule
Stochastic generator & \textbf{0.015} & 0.200

,"W1 (shape, population), rel. median","W1 (magnitude, population), rel. median"
method,,
Stochastic generator,0.015,0.200
Stochastic (bootstrap),0.017,0.011
"Best, mine",0.054,0.194
"Best, duration-corrected",0.054,0.194
Schedule-direct,0.060,0.064



% === process_2 ===
\begin{table}[H]
\caption{Aggregated Population-Level (unpaired) Distributional Evaluation for process\_2, experiment 951 (test evaluation). Real and method cases pooled per sensor (no case-to-case pairing), median over sensors. Shape: population generalization of the per-case W1-time metric (already scale-free). Magnitude: pooled raw-value W1 (sensor-type-agnostic, works for temperature too), relative to the real population\'s own value std. Lower is better. Rows sorted best-to-worst by shape. \textbf{Bold} = best method per column.}
\label{tab:population\_eval\_agg\_951\_test\_process\_2}
\begin{tabular}{l|c|c}
\toprule
 & W1 (shape, population), rel. median & W1 (magnitude, population), rel. median \\
method &  &  \\
\midrule
Stochastic (bootstrap) & \textbf{0.036} & \textbf{0.074} \\
Stochastic generator & 0.047 & 0.443 \\
Best, mine & 0.050 & 0.206 \\
Best, duration-corrected & 0.050 & 0.206 \\
Schedule-direct & 0.092 & 0.187 \\
\bottomrule
\end{tabular}
\end{

,"W1 (shape, population), rel. median","W1 (magnitude, population), rel. median"
method,,
Stochastic (bootstrap),0.036,0.074
Stochastic generator,0.047,0.443
"Best, mine",0.050,0.206
"Best, duration-corrected",0.050,0.206
Schedule-direct,0.092,0.187



% === process_3 ===
\begin{table}[H]
\caption{Aggregated Population-Level (unpaired) Distributional Evaluation for process\_3, experiment 951 (test evaluation). Real and method cases pooled per sensor (no case-to-case pairing), median over sensors. Shape: population generalization of the per-case W1-time metric (already scale-free). Magnitude: pooled raw-value W1 (sensor-type-agnostic, works for temperature too), relative to the real population\'s own value std. Lower is better. Rows sorted best-to-worst by shape. \textbf{Bold} = best method per column.}
\label{tab:population\_eval\_agg\_951\_test\_process\_3}
\begin{tabular}{l|c|c}
\toprule
 & W1 (shape, population), rel. median & W1 (magnitude, population), rel. median \\
method &  &  \\
\midrule
Stochastic (bootstrap) & \textbf{0.038} & \textbf{0.065} \\
Best, duration-corrected & 0.044 & 0.191 \\
Best, mine & 0.044 & 0.191 \\
Stochastic generator & 0.065 & 0.402 \\
Schedule-direct & 0.089 & 0.172 \\
\bottomrule
\end{tabular}
\end{

...

,"W1 (shape, population), rel. median","W1 (magnitude, population), rel. median"
method,,
Stochastic generator,0.001,0.516
Schedule-direct,0.002,0.855
Stochastic (bootstrap),0.002,0.561
"Best, mine",0.025,0.948
"Best, duration-corrected",0.025,0.948



% === process_4_2 ===
\begin{table}[H]
\caption{Aggregated Population-Level (unpaired) Distributional Evaluation for process\_4\_2, experiment 951 (test evaluation). Real and method cases pooled per sensor (no case-to-case pairing), median over sensors. Shape: population generalization of the per-case W1-time metric (already scale-free). Magnitude: pooled raw-value W1 (sensor-type-agnostic, works for temperature too), relative to the real population\'s own value std. Lower is better. Rows sorted best-to-worst by shape. \textbf{Bold} = best method per column.}
\label{tab:population\_eval\_agg\_951\_test\_process\_4\_2}
\begin{tabular}{l|c|c}
\toprule
 & W1 (shape, population), rel. median & W1 (magnitude, population), rel. median \\
method &  &  \\
\midrule
Stochastic generator & \textbf{0.001} & 0.910 \\
Stochastic (bootstrap) & 0.002 & 1.067 \\
Schedule-direct & 0.002 & 1.023 \\
Best, mine & 0.021 & \textbf{0.894} \\
Best, duration-corrected & 0.021 & \textbf{0.894} \\
\bottomrule
\e

,"W1 (shape, population), rel. median","W1 (magnitude, population), rel. median"
method,,
Stochastic generator,0.001,0.910
Stochastic (bootstrap),0.002,1.067
Schedule-direct,0.002,1.023
"Best, mine",0.021,0.894
"Best, duration-corrected",0.021,0.894



% === process_5 ===
\begin{table}[H]
\caption{Aggregated Population-Level (unpaired) Distributional Evaluation for process\_5, experiment 951 (test evaluation). Real and method cases pooled per sensor (no case-to-case pairing), median over sensors. Shape: population generalization of the per-case W1-time metric (already scale-free). Magnitude: pooled raw-value W1 (sensor-type-agnostic, works for temperature too), relative to the real population\'s own value std. Lower is better. Rows sorted best-to-worst by shape. \textbf{Bold} = best method per column.}
\label{tab:population\_eval\_agg\_951\_test\_process\_5}
\begin{tabular}{l|c|c}
\toprule
 & W1 (shape, population), rel. median & W1 (magnitude, population), rel. median \\
method &  &  \\
\midrule
Schedule-direct & \textbf{0.003} & 0.688 \\
Stochastic (bootstrap) & 0.003 & \textbf{0.402} \\
Stochastic generator & 0.003 & 0.600 \\
Best, mine & 0.003 & 0.682 \\
Best, duration-corrected & 0.003 & 0.682 \\
\bottomrule
\end{tabular}
\end{

,"W1 (shape, population), rel. median","W1 (magnitude, population), rel. median"
method,,
Schedule-direct,0.003,0.688
Stochastic (bootstrap),0.003,0.402
Stochastic generator,0.003,0.600
"Best, mine",0.003,0.682
"Best, duration-corrected",0.003,0.682


## Coper

In [15]:
# -- Aggregated across all processes: one row per method, one column per process + Overall --
if df_sp_relative_all.empty:
    print('Nothing to show -- df_sp_relative_all is empty.')
else:
    # Fair equal-weight rollup: per process, express each axis relative to the
    # stochastic floor and average the two (raw-pooling let the ~10x-larger
    # value scale dominate and hid the timing result). Then median across
    # processes. Keep the two raw axes' Overall too, so the timing win is
    # visible at a glance.
    _t = df_sp_relative_all.pivot_table(index='method', columns='process', values='w1_time_rel', aggfunc='median')
    _v = df_sp_relative_all.pivot_table(index='method', columns='process', values='w1_value_rel', aggfunc='median')
    _ft = _t.loc['Stochastic generator']; _fv = _v.loc['Stochastic generator']
    _comb = 0.5 * (_t.div(_ft, axis=1) + _v.div(_fv, axis=1))
    _rank_pivot = _comb.copy()
    _rank_pivot['W1(time) Overall']  = _t.median(axis=1)
    _rank_pivot['W1(value) Overall'] = _v.median(axis=1)
    _rank_pivot['Overall'] = _comb.median(axis=1)
    _rank_pivot = _rank_pivot.sort_values('Overall')

    _str_table = _bold_best_per_column(_rank_pivot)
    n_cols = len(_rank_pivot.columns)
    col_format = 'l|' + '|'.join(['c'] * (n_cols - 1)) + '|c'
    _rank_latex = _str_table.to_latex(
        escape=False, column_format=col_format,
        caption=(
            f'Aggregated Schedule Profile Evaluation across all processes, experiment {EXPERIMENT} ({SPLIT} evaluation). '
            r'Each case normalized by its own real curve scale (own duration for time, own mean/std for value); '
            r'each axis relative to the stochastic floor and averaged (equal weight) per process, then median over processes for Overall; W1(time)/W1(value) Overall are the raw per-axis medians. '
            r'Lower is better. Rows sorted best-to-worst by Overall. \textbf{Bold} = best method per column.'
        ),
        label=f'tab:schedule_profile_agg_overall_{EXPERIMENT}_{SPLIT}',
        position='H',
    )
    print(_rank_latex.replace('_', r'\_'))
    display(_rank_pivot.round(3))

\begin{table}[H]
\caption{Aggregated Schedule Profile Evaluation across all processes, experiment 951 (test evaluation). Each case normalized by its own real curve scale (own duration for time, own mean/std for value); each axis relative to the stochastic floor and averaged (equal weight) per process, then median over processes for Overall; W1(time)/W1(value) Overall are the raw per-axis medians. Lower is better. Rows sorted best-to-worst by Overall. \textbf{Bold} = best method per column.}
\label{tab:schedule\_profile\_agg\_overall\_951\_test}
\begin{tabular}{l|c|c|c|c|c|c|c|c|c}
\toprule
process & process\_1 & process\_2 & process\_3 & process\_4\_1 & process\_4\_2 & process\_5 & W1(time) Overall & W1(value) Overall & Overall \\
method &  &  &  &  &  &  &  &  &  \\
\midrule
Schedule-direct & 0.637 & 0.773 & \textbf{0.831} & \textbf{0.924} & \textbf{0.839} & \textbf{0.803} & 0.075 & 0.775 & \textbf{0.817} \\
Stochastic (bootstrap) & 0.711 & \textbf{0.674} & 0.843 & 0.983 & 0.903 & 0.8

process,process_1,process_2,process_3,process_4_1,process_4_2,process_5,W1(time) Overall,W1(value) Overall,Overall
method,,,,,,,,,
Schedule-direct,0.637,0.773,0.831,0.924,0.839,0.803,0.075,0.775,0.817
Stochastic (bootstrap),0.711,0.674,0.843,0.983,0.903,0.892,0.087,0.807,0.868
"Best, duration-corrected",0.778,0.725,0.956,1.769,1.547,0.950,0.068,0.685,0.953
Stochastic generator,1.000,1.000,1.000,1.000,1.000,1.000,0.093,1.029,1.000
"Best, mine",0.610,0.798,1.028,5.848,6.260,2.880,0.213,0.685,1.954


## Aggregated across all processes

Same three methods, one row each, now rolled up across every process: median over
sensors within a process (as above), then median over processes for `Overall`. Rows
sorted best-to-worst by `Overall`.


In [20]:
# -- Aggregated across all processes: one row per method, one column per process + Overall --
if df_sp_relative_all.empty:
    print('Nothing to show -- df_sp_relative_all is empty.')
else:
    _pooled_all = pd.concat([
        df_sp_relative_all[['process', 'method', 'w1_time_rel']].rename(columns={'w1_time_rel': 'v'}),
        df_sp_relative_all[['process', 'method', 'w1_value_rel']].rename(columns={'w1_value_rel': 'v'}),
    ], ignore_index=True)

    _rank_agg = _pooled_all.groupby(['process', 'method'])['v'].median().reset_index()
    _rank_pivot = _rank_agg.pivot(index='method', columns='process', values='v')
    _rank_pivot['Overall'] = _rank_pivot.median(axis=1)
    _rank_pivot = _rank_pivot.sort_values('Overall')

    _str_table = _bold_best_per_column(_rank_pivot)
    n_cols = len(_rank_pivot.columns)
    col_format = 'l|' + '|'.join(['c'] * (n_cols - 1)) + '|c'
    _rank_latex = _str_table.to_latex(
        escape=False, column_format=col_format,
        caption=(
            f'Aggregated Schedule Profile Evaluation across all processes, experiment {EXPERIMENT} ({SPLIT} evaluation). '
            r'Each case normalized by its own real curve scale (own duration for time, own mean/std for value); '
            r'both standardized metrics (W1 time, W1 value) pooled per process, then median over processes for Overall. '
            r'Lower is better. Rows sorted best-to-worst by Overall. \textbf{Bold} = best method per column.'
        ),
        label=f'tab:schedule_profile_agg_overall_{EXPERIMENT}_{SPLIT}',
        position='H',
    )
    print(_rank_latex.replace('_', r'\_'))
    display(_rank_pivot.round(3))


\begin{table}[H]
\caption{Aggregated Schedule Profile Evaluation across all processes, experiment 951 (test evaluation). Each case normalized by its own real curve scale (own duration for time, own mean/std for value); both standardized metrics (W1 time, W1 value) pooled per process, then median over processes for Overall. Lower is better. Rows sorted best-to-worst by Overall. \textbf{Bold} = best method per column.}
\label{tab:schedule\_profile\_agg\_overall\_951\_test}
\begin{tabular}{l|c|c|c|c|c|c|c}
\toprule
process & process\_1 & process\_2 & process\_3 & process\_4\_1 & process\_4\_2 & process\_5 & Overall \\
method &  &  &  &  &  &  &  \\
\midrule
Stochastic (bootstrap) & 0.117 & \textbf{0.140} & \textbf{0.195} & 0.379 & 0.243 & \textbf{0.321} & \textbf{0.219} \\
Schedule-direct & \textbf{0.105} & 0.198 & 0.204 & 0.407 & 0.340 & 0.554 & 0.272 \\
Best, duration-corrected & 0.149 & 0.193 & 0.216 & 0.356 & 0.332 & 0.468 & 0.274 \\
Stochastic generator & 0.169 & 0.294 & 0.281 & \tex

process,process_1,process_2,process_3,process_4_1,process_4_2,process_5,Overall
method,,,,,,,
Stochastic (bootstrap),0.117,0.140,0.195,0.379,0.243,0.321,0.219
Schedule-direct,0.105,0.198,0.204,0.407,0.340,0.554,0.272
"Best, duration-corrected",0.149,0.193,0.216,0.356,0.332,0.468,0.274
Stochastic generator,0.169,0.294,0.281,0.311,0.236,0.394,0.288
"Best, mine",0.131,0.206,0.237,0.394,0.387,0.538,0.312


### Aggregated across all processes -- both W1 metrics kept separate

Same idea as the table above, but instead of pooling W1 (time) and W1 (value) into one 'Combined'/'Overall' number, this keeps them as two separate columns: the median of `w1_time_rel` and the median of `w1_value_rel`, each taken over every (process, sensor) pair (using the same common-sensor filtering as above, so all methods are compared over the same sensors).

In [17]:
# -- Aggregated across all processes: both W1 metrics as separate columns (no pooling) --
if df_sp_relative_all.empty:
    print('Nothing to show -- df_sp_relative_all is empty.')
else:
    _both_agg = df_sp_relative_all.groupby('method').agg(
        **{'W1 (time), rel. median': ('w1_time_rel', 'median'),
           'W1 (value), rel. median': ('w1_value_rel', 'median')}
    )
    _both_agg = _both_agg.sort_values('W1 (time), rel. median')

    _both_str = _bold_best_per_column(_both_agg)
    _both_latex = _both_str.to_latex(
        escape=False, column_format='l|c|c',
        caption=(
            f'Aggregated Schedule Profile Evaluation across all processes, experiment {EXPERIMENT} ({SPLIT} evaluation). '
            r'Each case normalized by its own real curve scale (own duration for time, own mean/std for value). '
            r'W1 (time) and W1 (value), rel. -- each column is the median over every (process, sensor) pair, kept separate '
            r'(not pooled into a single score). Lower is better. \textbf{Bold} = best method per column.'
        ),
        label=f'tab:schedule_profile_agg_both_w1_{EXPERIMENT}_{SPLIT}',
        position='H',
    )
    print(_both_latex.replace('_', r'\_'))
    display(_both_agg.round(3))

\begin{table}[H]
\caption{Aggregated Schedule Profile Evaluation across all processes, experiment 951 (test evaluation). Each case normalized by its own real curve scale (own duration for time, own mean/std for value). W1 (time) and W1 (value), rel. -- each column is the median over every (process, sensor) pair, kept separate (not pooled into a single score). Lower is better. \textbf{Bold} = best method per column.}
\label{tab:schedule\_profile\_agg\_both\_w1\_951\_test}
\begin{tabular}{l|c|c}
\toprule
 & W1 (time), rel. median & W1 (value), rel. median \\
method &  &  \\
\midrule
Stochastic (bootstrap) & \textbf{0.061} & 1.573 \\
Stochastic generator & 0.061 & 1.839 \\
Schedule-direct & 0.061 & \textbf{1.276} \\
Best, duration-corrected & 0.062 & 1.485 \\
Best, mine & 0.226 & 1.485 \\
\bottomrule
\end{tabular}
\end{table}



,"W1 (time), rel. median","W1 (value), rel. median"
method,,
Stochastic (bootstrap),0.061,1.573
Stochastic generator,0.061,1.839
Schedule-direct,0.061,1.276
"Best, duration-corrected",0.062,1.485
"Best, mine",0.226,1.485


### Single flat median -- every case, every sensor, every process pooled together

No per-sensor or per-process grouping at all: every individual case instance (across all 6 processes and all their sensors) is thrown into one pool per method, and a single median is taken directly -- unlike the tables above, which take a median-of-medians (cases -> sensors -> processes, each level weighted equally regardless of how many cases/sensors it contains). Here, a process/sensor with more test cases contributes proportionally more to the result.

In [18]:
# -- Single flat median: pool every (process, sensor, case) instance together, no grouping --
if df_sp_cases_all.empty:
    print('Nothing to show -- df_sp_cases_all is empty.')
else:
    _flat = df_sp_cases_all.groupby('method').agg(
        **{'W1 (time), rel. median': ('w1_time_rel', 'median'),
           'W1 (value), rel. median': ('w1_value_rel', 'median'),
           'n_time': ('w1_time_rel', 'count'),
           'n_value': ('w1_value_rel', 'count')}
    )
    _flat_pooled = pd.concat([
        df_sp_cases_all[['method', 'w1_time_rel']].rename(columns={'w1_time_rel': 'v'}),
        df_sp_cases_all[['method', 'w1_value_rel']].rename(columns={'w1_value_rel': 'v'}),
    ], ignore_index=True)
    _flat['Combined, rel. median'] = _flat_pooled.groupby('method')['v'].median()
    _flat = _flat.sort_values('Combined, rel. median')

    _flat_display = _flat[['W1 (time), rel. median', 'W1 (value), rel. median', 'Combined, rel. median']]
    _flat_str = _bold_best_per_column(_flat_display)
    _flat_latex = _flat_str.to_latex(
        escape=False, column_format='l|c|c|c',
        caption=(
            f'Schedule Profile Evaluation, experiment {EXPERIMENT} ({SPLIT} evaluation), pooled flat across '
            r'every case instance from every sensor and every process (no per-sensor or per-process '
            r'grouping). Each case normalized by its own real curve scale (own duration for time, own '
            r'mean/std for value). Lower is better. \textbf{Bold} = best method per column.'
        ),
        label=f'tab:schedule_profile_flat_{EXPERIMENT}_{SPLIT}',
        position='H',
    )
    print(_flat_latex.replace('_', r'\_'))
    display(_flat.round(3))

\begin{table}[H]
\caption{Schedule Profile Evaluation, experiment 951 (test evaluation), pooled flat across every case instance from every sensor and every process (no per-sensor or per-process grouping). Each case normalized by its own real curve scale (own duration for time, own mean/std for value). Lower is better. \textbf{Bold} = best method per column.}
\label{tab:schedule\_profile\_flat\_951\_test}
\begin{tabular}{l|c|c|c}
\toprule
 & W1 (time), rel. median & W1 (value), rel. median & Combined, rel. median \\
method &  &  &  \\
\midrule
Stochastic (bootstrap) & 0.048 & 1.094 & \textbf{0.152} \\
Schedule-direct & 0.050 & \textbf{1.043} & 0.174 \\
Best, duration-corrected & 0.059 & 1.225 & 0.189 \\
Stochastic generator & \textbf{0.047} & 1.319 & 0.211 \\
Best, mine & 0.191 & 1.225 & 0.365 \\
\bottomrule
\end{tabular}
\end{table}



,"W1 (time), rel. median","W1 (value), rel. median",n_time,n_value,"Combined, rel. median"
method,,,,,
Stochastic (bootstrap),0.048,1.094,3125,2706,0.152
Schedule-direct,0.050,1.043,3196,2776,0.174
"Best, duration-corrected",0.059,1.225,3199,2779,0.189
Stochastic generator,0.047,1.319,3199,2779,0.211
"Best, mine",0.191,1.225,3199,2779,0.365


## Additional Population-Level Metrics

Three more axes, same real-vs-method pooled populations (no case pairing, strictly per process/sensor) as the shape/magnitude table above -- read `population_distribution_eval.csv` the same way, just different columns:

- **W1 (variability)** -- population W1 of per-case within-case std. Different from magnitude (which pools every raw reading with no notion of case at all): this catches methods that flatten real case-to-case volatility (regression-to-the-mean).
- **W1 (peak)** -- population W1 of per-case peak (max) value. Catches methods that get the average right but never reach real extremes.
- **Coverage** -- NOT a Wasserstein distance, a different kind of check: what fraction of real readings fall inside the method's own [5th, 95th] percentile band, pooled per sensor. **Higher is better** here (1.0 = every real reading is inside the method's plausible range) -- the opposite convention from every W1 column, which is lower-is-better.

In [19]:
# -- Additional Population-Level metrics: variability, peak, coverage --
# Same relative / floor-normalized / fairness-filter conventions as the shape+magnitude
# table above, reusing _sp_series and _sp_all_processes.

def _bold_best_per_column_dir(agg, higher_is_better_cols=()):
    """Like _bold_best_per_column, but a column can be marked higher-is-better
    (e.g. Coverage) instead of the default lower-is-better (every W1 metric)."""
    str_table = agg.round(3).astype(object)
    for col in agg.columns:
        vals = agg[col].dropna()
        if vals.empty:
            continue
        best_val = vals.max() if col in higher_is_better_cols else vals.min()
        for idx in agg.index:
            val = agg.loc[idx, col]
            if pd.isna(val):
                str_table.loc[idx, col] = ''
                continue
            fmt = f'{val:.3f}'
            str_table.loc[idx, col] = (r'\textbf{' + fmt + r'}') if abs(val - best_val) < 1e-6 else fmt
    return str_table


def _load_extra_population_eval(proc):
    """Returns [process, sensor, method, w1_variability_rel, w1_peak_rel, coverage]
    -- W1 axes relative to the real population's own scale for that sensor;
    coverage is already a 0..1 fraction, no scaling needed."""
    pop_path = run_dir / 'schedule_profile_eval_results' / proc / 'population_distribution_eval.csv'
    if not pop_path.exists():
        return pd.DataFrame()
    df_pop = pd.read_csv(pop_path)
    if 'w1_variability' not in df_pop.columns:
        return pd.DataFrame()

    rows = []
    for method, series_name in _sp_series.items():
        sub = df_pop[df_pop['method'] == series_name]
        for _, r in sub.iterrows():
            var_rel = (r['w1_variability'] / r['real_variability_scale']
                       if r['real_variability_scale'] > 1e-9 else float('nan'))
            peak_rel = (r['w1_peak'] / r['real_peak_scale']
                        if r['real_peak_scale'] > 1e-9 else float('nan'))
            rows.append({'process': proc, 'sensor': r['sensor'], 'method': method,
                        'w1_variability_rel': var_rel, 'w1_peak_rel': peak_rel,
                        'coverage': r.get('coverage', float('nan'))})
    return pd.DataFrame(rows)


df_extra_pop_all = pd.concat(
    [_load_extra_population_eval(p) for p in _sp_all_processes], ignore_index=True
) if _sp_all_processes else pd.DataFrame()

if not df_extra_pop_all.empty:
    _n_methods_expected_ep = len(_sp_series)
    _complete_counts_ep = (
        df_extra_pop_all.dropna(subset=['w1_variability_rel', 'w1_peak_rel', 'coverage'])
        .groupby(['process', 'sensor']).size()
    )
    _complete_keys_ep = set(_complete_counts_ep[_complete_counts_ep == _n_methods_expected_ep].index)
    _all_keys_ep = set(zip(df_extra_pop_all['process'], df_extra_pop_all['sensor']))
    _dropped_keys_ep = sorted(_all_keys_ep - _complete_keys_ep)
    if _dropped_keys_ep:
        print(f'[Extra Population Eval] Dropping {len(_dropped_keys_ep)} (process, sensor) pair(s) '
              f'lacking data for all {_n_methods_expected_ep} methods:')
        for _proc_k, _sensor_k in _dropped_keys_ep:
            print(f'    {_proc_k} / {_sensor_k}')
    df_extra_pop_all = df_extra_pop_all[
        df_extra_pop_all.set_index(['process', 'sensor']).index.isin(_complete_keys_ep)
    ].reset_index(drop=True)

for proc in _sp_all_processes:
    sub = df_extra_pop_all[df_extra_pop_all['process'] == proc] if not df_extra_pop_all.empty else pd.DataFrame()
    if sub.empty:
        print(f'{proc}: no additional population-eval data, skipping.')
        continue

    agg = sub.groupby('method').agg(
        **{'W1 (variability), rel. median': ('w1_variability_rel', 'median'),
           'W1 (peak), rel. median':        ('w1_peak_rel', 'median'),
           'Coverage, median':              ('coverage', 'median')}
    )
    agg = agg.sort_values('W1 (variability), rel. median')

    str_table = _bold_best_per_column_dir(agg, higher_is_better_cols=('Coverage, median',))
    latex = str_table.to_latex(
        escape=False, column_format='l|c|c|c',
        caption=(
            f'Additional Population-Level metrics for {proc}, experiment {EXPERIMENT} ({SPLIT} evaluation). '
            r'Real and method cases pooled per sensor (no case-to-case pairing), median over sensors. '
            r'W1 columns: lower is better. Coverage: fraction of real readings inside the method\'s own '
            r'5-95th percentile band -- HIGHER is better. \textbf{Bold} = best method per column.'
        ),
        label=f'tab:extra_population_eval_{EXPERIMENT}_{SPLIT}_{proc}',
        position='H',
    )
    print(f'% === {proc} ===')
    print(latex.replace('_', r'\_'))
    display(agg.round(3))
    print()


[Extra Population Eval] Dropping 2 (process, sensor) pair(s) lacking data for all 5 methods:
    process_1 / water_supply_power_kW_energy_to_model
    process_4_2 / (14)_filter_mas_kg/h_energy_to_model
% === process_1 ===
\begin{table}[H]
\caption{Additional Population-Level metrics for process\_1, experiment 951 (test evaluation). Real and method cases pooled per sensor (no case-to-case pairing), median over sensors. W1 columns: lower is better. Coverage: fraction of real readings inside the method\'s own 5-95th percentile band -- HIGHER is better. \textbf{Bold} = best method per column.}
\label{tab:extra\_population\_eval\_951\_test\_process\_1}
\begin{tabular}{l|c|c|c}
\toprule
 & W1 (variability), rel. median & W1 (peak), rel. median & Coverage, median \\
method &  &  &  \\
\midrule
Stochastic (bootstrap) & \textbf{0.185} & \textbf{0.475} & 0.950 \\
Stochastic generator & 0.683 & 3.220 & \textbf{0.964} \\
Best, mine & 0.744 & 2.061 & 0.950 \\
Best, duration-corrected & 0.744 & 2.06

,"W1 (variability), rel. median","W1 (peak), rel. median","Coverage, median"
method,,,
Stochastic (bootstrap),0.185,0.475,0.950
Stochastic generator,0.683,3.220,0.964
"Best, mine",0.744,2.061,0.950
"Best, duration-corrected",0.744,2.061,0.950
Schedule-direct,0.906,1.209,0.915



% === process_2 ===
\begin{table}[H]
\caption{Additional Population-Level metrics for process\_2, experiment 951 (test evaluation). Real and method cases pooled per sensor (no case-to-case pairing), median over sensors. W1 columns: lower is better. Coverage: fraction of real readings inside the method\'s own 5-95th percentile band -- HIGHER is better. \textbf{Bold} = best method per column.}
\label{tab:extra\_population\_eval\_951\_test\_process\_2}
\begin{tabular}{l|c|c|c}
\toprule
 & W1 (variability), rel. median & W1 (peak), rel. median & Coverage, median \\
method &  &  &  \\
\midrule
Stochastic (bootstrap) & \textbf{0.277} & \textbf{0.291} & \textbf{0.958} \\
Best, duration-corrected & 0.743 & 0.849 & 0.890 \\
Best, mine & 0.743 & 0.849 & 0.890 \\
Stochastic generator & 0.906 & 0.803 & 0.930 \\
Schedule-direct & 1.521 & 0.874 & 0.912 \\
\bottomrule
\end{tabular}
\end{table}



,"W1 (variability), rel. median","W1 (peak), rel. median","Coverage, median"
method,,,
Stochastic (bootstrap),0.277,0.291,0.958
"Best, duration-corrected",0.743,0.849,0.890
"Best, mine",0.743,0.849,0.890
Stochastic generator,0.906,0.803,0.930
Schedule-direct,1.521,0.874,0.912



% === process_3 ===
\begin{table}[H]
\caption{Additional Population-Level metrics for process\_3, experiment 951 (test evaluation). Real and method cases pooled per sensor (no case-to-case pairing), median over sensors. W1 columns: lower is better. Coverage: fraction of real readings inside the method\'s own 5-95th percentile band -- HIGHER is better. \textbf{Bold} = best method per column.}
\label{tab:extra\_population\_eval\_951\_test\_process\_3}
\begin{tabular}{l|c|c|c}
\toprule
 & W1 (variability), rel. median & W1 (peak), rel. median & Coverage, median \\
method &  &  &  \\
\midrule
Stochastic (bootstrap) & \textbf{0.483} & \textbf{0.698} & 0.950 \\
Best, duration-corrected & 0.627 & 0.949 & \textbf{0.961} \\
Best, mine & 0.627 & 0.949 & \textbf{0.961} \\
Stochastic generator & 0.791 & 0.838 & 0.946 \\
Schedule-direct & 1.356 & 0.835 & 0.924 \\
\bottomrule
\end{tabular}
\end{table}



...

,"W1 (variability), rel. median","W1 (peak), rel. median","Coverage, median"
method,,,
Stochastic (bootstrap),0.539,0.837,0.926
Schedule-direct,1.372,0.994,0.230
"Best, mine",1.527,1.342,0.093
"Best, duration-corrected",1.527,1.342,0.093
Stochastic generator,2.958,3.768,0.956



% === process_4_2 ===
\begin{table}[H]
\caption{Additional Population-Level metrics for process\_4\_2, experiment 951 (test evaluation). Real and method cases pooled per sensor (no case-to-case pairing), median over sensors. W1 columns: lower is better. Coverage: fraction of real readings inside the method\'s own 5-95th percentile band -- HIGHER is better. \textbf{Bold} = best method per column.}
\label{tab:extra\_population\_eval\_951\_test\_process\_4\_2}
\begin{tabular}{l|c|c|c}
\toprule
 & W1 (variability), rel. median & W1 (peak), rel. median & Coverage, median \\
method &  &  &  \\
\midrule
Stochastic (bootstrap) & \textbf{0.221} & 0.963 & 0.576 \\
Best, duration-corrected & 0.451 & 1.678 & 0.023 \\
Best, mine & 0.451 & 1.678 & 0.023 \\
Schedule-direct & 0.475 & \textbf{0.954} & 0.279 \\
Stochastic generator & 0.926 & 1.491 & \textbf{0.830} \\
\bottomrule
\end{tabular}
\end{table}



,"W1 (variability), rel. median","W1 (peak), rel. median","Coverage, median"
method,,,
Stochastic (bootstrap),0.221,0.963,0.576
"Best, duration-corrected",0.451,1.678,0.023
"Best, mine",0.451,1.678,0.023
Schedule-direct,0.475,0.954,0.279
Stochastic generator,0.926,1.491,0.830



% === process_5 ===
\begin{table}[H]
\caption{Additional Population-Level metrics for process\_5, experiment 951 (test evaluation). Real and method cases pooled per sensor (no case-to-case pairing), median over sensors. W1 columns: lower is better. Coverage: fraction of real readings inside the method\'s own 5-95th percentile band -- HIGHER is better. \textbf{Bold} = best method per column.}
\label{tab:extra\_population\_eval\_951\_test\_process\_5}
\begin{tabular}{l|c|c|c}
\toprule
 & W1 (variability), rel. median & W1 (peak), rel. median & Coverage, median \\
method &  &  &  \\
\midrule
Stochastic (bootstrap) & \textbf{0.635} & \textbf{0.634} & 0.799 \\
Schedule-direct & 0.966 & 0.680 & 0.288 \\
Best, mine & 0.979 & 0.879 & 0.472 \\
Best, duration-corrected & 0.979 & 0.879 & 0.472 \\
Stochastic generator & 3.166 & 2.971 & \textbf{0.972} \\
\bottomrule
\end{tabular}
\end{table}



,"W1 (variability), rel. median","W1 (peak), rel. median","Coverage, median"
method,,,
Stochastic (bootstrap),0.635,0.634,0.799
Schedule-direct,0.966,0.680,0.288
"Best, mine",0.979,0.879,0.472
"Best, duration-corrected",0.979,0.879,0.472
Stochastic generator,3.166,2.971,0.972
